# Bài 6.3 · BERT tiếng Việt trên dữ liệu thật

**UIT-VSFC** gồm hơn 16 000 câu phản hồi của sinh viên về môn học và giảng viên, gán nhãn cảm xúc tiêu cực, trung tính, tích cực. Nhãn trung tính chỉ chiếm khoảng 4%, nên metric là **macro-F1**: mỗi lớp góp phần ngang nhau, lớp hiếm không bị lấn át.

Notebook đi theo thứ tự của bài:

1. baseline TF-IDF và hồi quy logistic (bài 6.1),
2. tách từ tiếng Việt và tokenizer của **PhoBERT**,
3. fine-tune PhoBERT bằng một vòng huấn luyện tự viết, chọn epoch theo validation,
4. chấm test một lần, đọc ma trận nhầm lẫn, thử vài câu tự viết.

Cần bật **GPU** và **Internet** trong *Session options*. Chạy hết khoảng 8 phút.

In [ ]:
!pip install -q pyvi

In [ ]:
import time, copy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, transformers
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
from pyvi import ViTokenizer

torch.manual_seed(0); np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| thiết bị:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')
LABELS = ['tiêu cực', 'trung tính', 'tích cực']

## 1. Đọc dữ liệu

Ba tập train, validation, test đã chia sẵn, tải dạng parquet từ Hugging Face. Cột `sentiment` là nhãn: 0 tiêu cực, 1 trung tính, 2 tích cực.

In [ ]:
URL = 'https://huggingface.co/api/datasets/uitnlp/vietnamese_students_feedback/parquet/default/{}/0.parquet'
df = {s: pd.read_parquet(URL.format(s)) for s in ['train', 'validation', 'test']}
print({s: len(d) for s, d in df.items()})
print(df['train']['sentiment'].value_counts(normalize=True).rename(index=dict(enumerate(LABELS))).round(3).to_dict())
df['train'].head()

## 2. Baseline: TF-IDF và hồi quy logistic

Đếm từ và cặp từ, đánh trọng số TF-IDF, rồi phân loại tuyến tính. `class_weight='balanced'` tăng trọng số lớp trung tính hiếm. Đây là mốc mà PhoBERT phải vượt.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = tfidf.fit_transform(df['train'].sentence); Xte = tfidf.transform(df['test'].sentence)
lr = LogisticRegression(max_iter=3000, class_weight='balanced').fit(Xtr, df['train'].sentiment)
f1_tfidf = f1_score(df['test'].sentiment, lr.predict(Xte), average='macro')
print(f'TF-IDF + logistic: macro-F1 test = {f1_tfidf:.4f}')

## 3. Tách từ tiếng Việt và tokenizer của PhoBERT

Tiếng Việt viết cách theo âm tiết: "giảng viên" là một từ hai âm tiết. PhoBERT học trên văn bản đã tách từ, các âm tiết của một từ nối bằng dấu gạch dưới. `pyvi` làm bước này, rồi tokenizer BPE của PhoBERT cắt tiếp thành subword.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base')
s = 'giảng viên nhiệt tình , bài giảng dễ hiểu nhưng phòng học hơi nóng .'
seg = ViTokenizer.tokenize(s)
print('câu gốc :', s); print('tách từ :', seg)
print('token   :', tokenizer.tokenize(seg))
print('id      :', tokenizer(seg)['input_ids'])

In [ ]:
t = time.time()
seg = {k: [ViTokenizer.tokenize(x) for x in d.sentence] for k, d in df.items()}
lens = [len(tokenizer(x)['input_ids']) for x in seg['train']]
print(f'tách từ xong sau {time.time() - t:.0f} giây | số token: trung vị {int(np.median(lens))}, phân vị 99% {int(np.percentile(lens, 99))}, lớn nhất {max(lens)}')
MAX_LEN = 96

## 4. Fine-tune PhoBERT

Mỗi batch chỉ đệm tới câu dài nhất của batch đó (`tokenizer.pad`), nên không tốn tính toán cho token đệm. Learning rate tăng dần trong 10% bước đầu rồi giảm tuyến tính. Sau mỗi epoch chấm macro-F1 trên validation và giữ bản tốt nhất; test chưa được đụng tới.

In [ ]:
def make_loader(split, shuffle, bs=32):
    enc = tokenizer(seg[split], truncation=True, max_length=MAX_LEN)
    items = [{'input_ids': i, 'attention_mask': a, 'labels': int(y)}
             for i, a, y in zip(enc['input_ids'], enc['attention_mask'], df[split].sentiment)]
    return DataLoader(items, batch_size=bs, shuffle=shuffle, collate_fn=lambda b: tokenizer.pad(b, return_tensors='pt'))

loaders = {s: make_loader(s, s == 'train') for s in df}
model = AutoModelForSequenceClassification.from_pretrained('vinai/phobert-base', num_labels=3).to(device)
EPOCHS = 3
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
steps = EPOCHS * len(loaders['train'])
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')

def predict(loader):
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
        for b in loader:
            b = {k: v.to(device) for k, v in b.items() if k != 'labels'}
            out.append(model(**b).logits.argmax(-1).cpu())
    return torch.cat(out).numpy()

best_f1, best_state = -1, None
for ep in range(EPOCHS):
    t = time.time(); model.train()
    for b in loaders['train']:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
            loss = model(**b).loss
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
    f1 = f1_score(df['validation'].sentiment, predict(loaders['validation']), average='macro')
    print(f'epoch {ep + 1}: loss cuối {loss.item():.3f} | macro-F1 validation {f1:.4f} | {time.time() - t:.0f} giây')
    if f1 > best_f1:
        best_f1, best_state = f1, copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
model.load_state_dict(best_state)

## 5. Chấm test một lần

Bản tốt nhất theo validation được chấm trên test. Ma trận nhầm lẫn cho thấy lỗi tập trung ở đâu, nhất là với lớp trung tính hiếm.

In [ ]:
pred = predict(loaders['test'])
f1_bert = f1_score(df['test'].sentiment, pred, average='macro')
print(f'macro-F1 test: TF-IDF {f1_tfidf:.4f} → PhoBERT {f1_bert:.4f}')
print(classification_report(df['test'].sentiment, pred, target_names=LABELS, digits=3))
ConfusionMatrixDisplay.from_predictions(df['test'].sentiment, pred, display_labels=LABELS, cmap='Oranges')
plt.show()

In [ ]:
mine = ['thầy giảng rất dễ hiểu và nhiệt tình',
        'phòng học nóng quá , máy chiếu hay bị hỏng',
        'môn này có 3 tín chỉ',
        'bài tập nhiều nhưng học được nhiều thứ']
enc = tokenizer([ViTokenizer.tokenize(x) for x in mine], padding=True, return_tensors='pt').to(device)
model.eval()
with torch.no_grad():
    prob = model(**enc).logits.softmax(-1).cpu().numpy()
pd.DataFrame(prob.round(3), index=mine, columns=LABELS)

## Đọc kết quả

- PhoBERT vượt baseline TF-IDF rõ rệt về macro-F1. Phần lớn khoảng cách nằm ở lớp trung tính: mô hình hiểu câu chứ không chỉ đếm từ khen chê.
- Lớp trung tính vẫn là lớp khó nhất vì ít mẫu và ranh giới mờ. Muốn cải thiện thì thử trọng số lớp trong loss hoặc gộp dự đoán của nhiều seed, đo bằng macro-F1 trên validation.
- Bỏ bước tách từ (đưa thẳng câu gốc vào tokenizer) thường làm điểm giảm: input lệch với cách PhoBERT đã học.

Gợi ý khi demo: thêm một câu khen chê lẫn lộn vào danh sách `mine` và xem xác suất ba lớp thay đổi thế nào.